## What is DPO and Why Do We Need a Special Dataset?

| Phase | What the model sees | What it learns |
|---|---|---|
| SFT (Phase 1) | correct output only | *how* to extract JSON |
| DPO (Phase 2) | correct vs incorrect output | *why* one output is better than another |

DPO teaches the model **preferences**, not just patterns. For each input:
- ✅ `chosen` = the correct, complete, clean JSON extraction
- ❌ `rejected` = a realistic but flawed extraction (missing fields, wrong keys, etc.)

The model learns: *'When I see this input, I should produce chosen, NOT rejected.'*

## 11 Rejection Strategies Used
| # | Strategy | Example |
|---|---|---|
| 1 | **Missing fields** | Drops 1-2 fields from the output |
| 2 | **Wrong key names** | `job` → `role`, `age` → `years_old` |
| 3 | **Wrong values** | Corrupts a value slightly |
| 4 | **Hallucinated fields** | Adds fake fields like `department`, `priority` |
| 5 | **Partial extraction** | Only extracts 1 field from a 5-field input |
| 6 | **Type errors** | Numbers as strings: `age: "33"` instead of `33` |
| 7 | **Invalid JSON** | Malformed JSON the model shouldn't produce |
| 8 | **Extra text wrapper** | `Here is the info: {...}` (not pure JSON) |
| 9 | **Swapped values** | `name` and `company` values swapped |
| 10 | **Empty values** | `"city": ""` or `"age": null` |
| 11 | **Nested wrong** | `"job": {"value": "engineer"}` instead of string |


---
## Step 1 — Imports and Setup

In [1]:
import json, random, copy, os
from pathlib import Path
from collections import Counter

random.seed(42)

BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "data")

DATA_DIR = '../data'          # Where your SFT .jsonl files are
OUT_DIR  = '../data'          # Where DPO files will be saved

# Colab example:  DATA_DIR = '/content/llm-finetune-project/data'
# Windows example: DATA_DIR = r'C:\Users\YourName\llm-finetune-project\data'

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'✅ Input dir:  {os.path.abspath(DATA_DIR)}')
print(f'✅ Output dir: {os.path.abspath(OUT_DIR)}')


✅ Input dir:  c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\data
✅ Output dir: c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\data


---
## Step 2 — Load Your SFT Dataset

We build DPO pairs directly from your existing `train.jsonl`, `val.jsonl`, `test.jsonl`.

In [2]:
def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f'  ⚠️  Skipping line {i}: {e}')
    return rows

train_sft = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val_sft   = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))
test_sft  = load_jsonl(os.path.join(DATA_DIR, 'test.jsonl'))

print(f'✅ Loaded SFT data:')
print(f'   train.jsonl: {len(train_sft)} examples')
print(f'   val.jsonl:   {len(val_sft)} examples')
print(f'   test.jsonl:  {len(test_sft)} examples')
print(f'   TOTAL:       {len(train_sft)+len(val_sft)+len(test_sft)} examples')

# Preview
print('\nSample SFT row (what we start with):')
s = train_sft[0]
print(f'  instruction: {s["instruction"]}')
print(f'  input:       {s["input"]}')
print(f'  output:      {s["output"]}')


✅ Loaded SFT data:
   train.jsonl: 1616 examples
   val.jsonl:   202 examples
   test.jsonl:  203 examples
   TOTAL:       2021 examples

Sample SFT row (what we start with):
  instruction: Extract entities and values from the text as JSON only.
  input:       Aaron Mehta is a 37-year-old civil engineer living in Lima.
  output:      {"name": "Aaron Mehta", "age": 37, "job": "civil engineer", "city": "Lima"}


---
## Step 3 — Define All 11 Rejection Strategies

Each strategy simulates a **different type of real model mistake**.
This teaches the model to avoid each category of error separately.


In [3]:
# ═══════════════════════════════════════════════════════════════
# KEY TYPO MAP — wrong names a bad model might use
# ═══════════════════════════════════════════════════════════════
KEY_TYPOS = {
    "name":           ["full_name","Name","names","user_name","person","username"],
    "age":            ["Age","years","user_age","years_old","birth_age","yr"],
    "city":           ["City","location","place","town","address","loc"],
    "job":            ["Job","role","occupation","position","work","title"],
    "company":        ["Company","employer","firm","org","organization","corp"],
    "email":          ["Email","mail","e_mail","emailAddress","email_id"],
    "phone":          ["Phone","mobile","contact","telephone","number","cell"],
    "order_id":       ["orderId","order","id","orderNumber","order_number","ref"],
    "product":        ["Product","item","goods","productName","prod","article"],
    "quantity":       ["Quantity","qty","count","num","units","pieces","no"],
    "price_per_unit": ["price","unit_price","cost","unitPrice","per_unit","rate"],
    "total":          ["Total","sum","totalAmount","total_cost","grand_total","bill"],
    "color":          ["Color","colour","shade","product_color","hue"],
    "customer":       ["Customer","client","buyer","user","purchaser","shopper"],
    "event":          ["Event","meeting","title","type","event_type","subject"],
    "date":           ["Date","day","event_date","scheduled_date","on","when"],
    "time":           ["Time","hour","start_time","at","clock","starts"],
    "location":       ["Location","place","venue","room","where","site"],
    "host":           ["Host","organizer","presenter","owner","hosted_by","by"],
    "duration_minutes":["duration","minutes","length","time_length","dur","mins"],
    "invoice_id":     ["invoiceId","invoice","inv_id","invoice_number","inv_no"],
    "amount":         ["Amount","value","sum","price","cost","total_amount"],
    "currency":       ["Currency","curr","money_type","unit","cur","coin"],
    "status":         ["Status","state","condition","payment_status","flag","stage"],
    "due_day":        ["dueDay","due","due_date","payment_day","by","deadline"],
    "from":           ["From","origin","source","departure","from_city","src"],
    "to":             ["To","destination","dest","arrival","to_city","dst"],
    "passengers":     ["Passengers","people","travelers","pax","seats","count"],
    "airline":        ["Airline","carrier","air","flight_operator","airways","carrier_name"],
    "class":          ["Class","seat_class","ticket_class","cabin","tier","category"],
    "traveler":       ["Traveler","passenger","booker","traveller","flyer","guest"],
    "patient":        ["Patient","person","user","client","name","individual"],
    "diagnosis":      ["Diagnosis","disease","illness","condition","ailment","problem"],
    "condition":      ["Condition","disease","illness","ailment","diagnosis","issue"],
    "medication":     ["Medication","medicine","drug","med","prescription","rx"],
    "dosage":         ["Dosage","dose","amount","strength","quantity","mg"],
    "frequency":      ["Frequency","how_often","schedule","interval","times","regime"],
}

# Fake fields a hallucinating model might add
FAKE_FIELDS = [
    ("department", "Engineering"), ("priority", "high"), ("notes", "N/A"),
    ("reference", "REF-0001"), ("approved_by", "Admin"), ("category", "general"),
    ("tax", 0.0), ("discount", "10%"), ("region", "North"), ("team", "Alpha"),
    ("source", "web"), ("type", "standard"), ("tier", "basic"), ("flag", False),
    ("created_at", "2025-01-01"), ("updated_at", "2025-01-02"), ("version", 1),
    ("id", 9999), ("score", 0.95), ("label", "auto-generated"),
    ("remarks", "none"), ("verified", True), ("tag", "new"), ("group", "default"),
]

print('✅ Rejection vocabulary defined')
print(f'   Key typo map: {len(KEY_TYPOS)} fields covered')
print(f'   Fake fields pool: {len(FAKE_FIELDS)} options')


✅ Rejection vocabulary defined
   Key typo map: 37 fields covered
   Fake fields pool: 24 options


In [4]:
def apply_rejection(output_dict, strategy):
    """
    Apply one rejection strategy to a correct output dict.
    Returns a string (the rejected JSON or text).
    The returned string is ALWAYS different from json.dumps(output_dict).
    """
    rejected = copy.deepcopy(output_dict)

    # ── STRATEGY 1: Missing fields ────────────────────────────
    if strategy == 'missing_fields':
        keys = list(rejected.keys())
        if len(keys) >= 2:
            n_remove = random.randint(1, max(1, len(keys) // 2))
            for k in random.sample(keys, n_remove):
                del rejected[k]
        else:
            # Only 1 field — remove it entirely (return empty dict)
            rejected = {}

    # ── STRATEGY 2: Wrong key names ───────────────────────────
    elif strategy == 'wrong_key_names':
        keys = list(rejected.keys())
        # Rename ALL renameable keys
        n_rename = random.randint(1, min(2, len(keys)))
        for k in random.sample(keys, n_rename):
            if k in KEY_TYPOS:
                wrong_key = random.choice(KEY_TYPOS[k])
                rejected[wrong_key] = rejected.pop(k)

    # ── STRATEGY 3: Wrong values ──────────────────────────────
    elif strategy == 'wrong_values':
        keys = list(rejected.keys())
        k = random.choice(keys)
        v = rejected[k]
        if isinstance(v, int):
            rejected[k] = v + random.choice([-5, -10, 5, 10, 50, 100, -1, 1, -100])
        elif isinstance(v, float):
            rejected[k] = round(v * random.choice([0.5, 1.5, 2.0, 0.1]), 2)
        elif isinstance(v, str) and len(v) > 2:
            # Corrupt the string
            corrupt_type = random.choice(['delete_char', 'swap_chars', 'add_suffix', 'truncate'])
            if corrupt_type == 'delete_char':
                idx = random.randint(1, len(v)-2)
                rejected[k] = v[:idx] + v[idx+1:]
            elif corrupt_type == 'swap_chars' and len(v) >= 4:
                i = random.randint(1, len(v)-2)
                lst = list(v)
                lst[i], lst[i-1] = lst[i-1], lst[i]
                rejected[k] = ''.join(lst)
            elif corrupt_type == 'add_suffix':
                rejected[k] = v + random.choice(['_old','_temp',' Jr.',' II',' (unverified)'])
            else:  # truncate
                rejected[k] = v[:max(2, len(v)//2)]
        else:
            rejected[k] = str(v) + '_invalid'

    # ── STRATEGY 4: Hallucinated fields ──────────────────────
    elif strategy == 'hallucinated_fields':
        n_add = random.randint(1, 3)
        for fake_k, fake_v in random.sample(FAKE_FIELDS, min(n_add, len(FAKE_FIELDS))):
            if fake_k not in rejected:
                rejected[fake_k] = fake_v

    # ── STRATEGY 5: Partial extraction ───────────────────────
    elif strategy == 'partial_extraction':
        keys = list(rejected.keys())
        if len(keys) >= 3:
            keep_n = random.randint(1, max(1, len(keys) - 2))
            keep = random.sample(keys, keep_n)
            rejected = {k: rejected[k] for k in keep}
        elif len(keys) == 2:
            keep = [random.choice(keys)]
            rejected = {k: rejected[k] for k in keep}

    # ── STRATEGY 6: Type errors ───────────────────────────────
    elif strategy == 'type_errors':
        for k in list(rejected.keys()):
            if isinstance(rejected[k], (int, float)):
                # Number → string (wrong type)
                rejected[k] = str(rejected[k])
                break

    # ── STRATEGY 7: Invalid JSON string ──────────────────────
    elif strategy == 'invalid_json_string':
        good = json.dumps(rejected)
        corrupt = random.choice([
            good[:-1],                            # missing closing brace
            good.replace('":', ' ='),             # colon → equals
            good.replace(',', ',,'),              # double comma
            good.replace('{', '').replace('}',''), # no braces
            good.replace('"', "'"),               # single quotes (invalid JSON)
            '{' + good[2:],                       # missing first quote
        ])
        return corrupt  # Return as-is (not re-serialized)

    # ── STRATEGY 8: Extra text wrapper ───────────────────────
    elif strategy == 'extra_text_wrapper':
        prefix = random.choice([
            'Here is the extracted information: ',
            'Based on the input, I found: ',
            'The extracted data is as follows: ',
            'After analyzing the text: ',
            'Extraction result: ',
        ])
        suffix = random.choice([
            '. Let me know if you need more details.',
            '. Hope this helps!',
            '.',
            ' Please verify the information.',
        ])
        return prefix + json.dumps(rejected) + suffix

    # ── STRATEGY 9: Swapped values ────────────────────────────
    elif strategy == 'swapped_values':
        keys = list(rejected.keys())
        if len(keys) >= 2:
            k1, k2 = random.sample(keys, 2)
            v1, v2 = rejected[k1], rejected[k2]
            # Only swap if same type (makes sense as a realistic mistake)
            if type(v1) == type(v2) and k1 != k2:
                rejected[k1], rejected[k2] = v2, v1
            else:
                # Force swap anyway for string fields
                rejected[k1] = str(v2)
                rejected[k2] = str(v1)

    # ── STRATEGY 10: Empty values ─────────────────────────────
    elif strategy == 'empty_values':
        keys = list(rejected.keys())
        # Set 1-2 fields to empty/null
        n_empty = random.randint(1, min(2, len(keys)))
        for k in random.sample(keys, n_empty):
            if isinstance(rejected[k], str):
                rejected[k] = random.choice(['', 'N/A', 'unknown', 'null', 'undefined'])
            else:
                rejected[k] = None

    # ── STRATEGY 11: Nested wrong ────────────────────────────
    elif strategy == 'nested_wrong':
        keys = list(rejected.keys())
        k = random.choice(keys)
        nest_type = random.choice(['value_wrap', 'array_wrap', 'object_group'])
        if nest_type == 'value_wrap':
            rejected[k] = {'value': rejected[k]}
        elif nest_type == 'array_wrap':
            rejected[k] = [rejected[k]]
        else:  # object_group — wraps all into a sub-object
            if len(keys) >= 2:
                group_keys = random.sample(keys, random.randint(2, min(3, len(keys))))
                sub = {k2: rejected.pop(k2) for k2 in group_keys if k2 in rejected}
                rejected['extracted'] = sub

    return json.dumps(rejected)


# All available strategies
ALL_STRATEGIES = [
    'missing_fields',
    'wrong_key_names',
    'wrong_values',
    'hallucinated_fields',
    'partial_extraction',
    'type_errors',
    'invalid_json_string',
    'extra_text_wrapper',
    'swapped_values',
    'empty_values',
    'nested_wrong',
]

print(f'✅ {len(ALL_STRATEGIES)} rejection strategies defined')

# ── Quick visual test ──
test_input  = 'Hassan Martinez, 33, pharmacist at Infosys in Delhi.'
test_chosen = {"name": "Hassan Martinez", "age": 33, "job": "pharmacist", "company": "Infosys", "city": "Delhi"}

print('\n🔬 Strategy test results:')
print(f'  CHOSEN:  {json.dumps(test_chosen)}')
print()
for s in ALL_STRATEGIES:
    r = apply_rejection(test_chosen, s)
    changed = '✅' if r != json.dumps(test_chosen) else '⚠️ SAME AS CHOSEN'
    print(f'  [{s:<25}] {changed}  →  {r[:90]}')


✅ 11 rejection strategies defined

🔬 Strategy test results:
  CHOSEN:  {"name": "Hassan Martinez", "age": 33, "job": "pharmacist", "company": "Infosys", "city": "Delhi"}

  [missing_fields           ] ✅  →  {"age": 33, "job": "pharmacist", "company": "Infosys", "city": "Delhi"}
  [wrong_key_names          ] ✅  →  {"name": "Hassan Martinez", "job": "pharmacist", "company": "Infosys", "years": 33, "loc":
  [wrong_values             ] ✅  →  {"name": "Hassan Marinez", "age": 33, "job": "pharmacist", "company": "Infosys", "city": "
  [hallucinated_fields      ] ✅  →  {"name": "Hassan Martinez", "age": 33, "job": "pharmacist", "company": "Infosys", "city": 
  [partial_extraction       ] ✅  →  {"age": 33}
  [type_errors              ] ✅  →  {"name": "Hassan Martinez", "age": "33", "job": "pharmacist", "company": "Infosys", "city"
  [invalid_json_string      ] ✅  →  {"name = "Hassan Martinez", "age = 33, "job = "pharmacist", "company = "Infosys", "city = 
  [extra_text_wrapper       ] ✅  →  Ex

---
## Step 4 — DPO Pair Generator

For each SFT example, this function:
1. Tries multiple rejection strategies
2. Picks one that **actually produces a different output** (no false negatives)
3. Ensures the chosen is always genuinely better than rejected
4. Rotates strategy usage so the distribution is even across the dataset


In [5]:
def make_dpo_pair(sft_example, strategy=None, max_retries=5):
    """
    Convert one SFT example into a DPO pair.
    Returns dict with: prompt, chosen, rejected, strategy_used
    Returns None if we can't create a valid different rejection.
    """
    try:
        chosen_dict = json.loads(sft_example['output'])
    except:
        return None  # skip malformed outputs

    chosen_str = json.dumps(chosen_dict)
    prompt = sft_example['input']

    # Try strategies until we get one that actually changes the output
    tried = set()
    for attempt in range(max_retries):
        if strategy:
            strat = strategy
        else:
            remaining = [s for s in ALL_STRATEGIES if s not in tried]
            if not remaining:
                break
            strat = random.choice(remaining)
        tried.add(strat)

        # Special case: skip some strategies for empty outputs (refusal cases)
        if not chosen_dict and strat not in ['hallucinated_fields', 'extra_text_wrapper', 'invalid_json_string']:
            continue

        rejected_str = apply_rejection(chosen_dict, strat)

        # Verify it's actually different
        if rejected_str != chosen_str:
            return {
                'prompt': prompt,
                'chosen': chosen_str,
                'rejected': rejected_str,
                'strategy_used': strat,   # useful for analysis; remove before training if desired
            }

    return None  # couldn't make a valid pair


def generate_dpo_dataset(sft_examples, n_per_example=1, strategy_rotation=True):
    """
    Convert a full SFT split into DPO pairs.

    Args:
        sft_examples: list of SFT dicts
        n_per_example: how many DPO pairs per SFT example (1 = same size as SFT)
        strategy_rotation: if True, rotate through strategies evenly

    Returns:
        list of DPO dicts, strategy_counter
    """
    dpo_pairs = []
    skipped = 0
    strategy_counter = Counter()

    # Build a strategy rotation cycle for even distribution
    strategy_cycle = ALL_STRATEGIES * (len(sft_examples) // len(ALL_STRATEGIES) + 2)
    random.shuffle(strategy_cycle)
    cycle_idx = 0

    for ex in sft_examples:
        for _ in range(n_per_example):
            if strategy_rotation:
                strat = strategy_cycle[cycle_idx % len(strategy_cycle)]
                cycle_idx += 1
                pair = make_dpo_pair(ex, strategy=strat)
                if pair is None:  # strategy didn't work, try random
                    pair = make_dpo_pair(ex, strategy=None)
            else:
                pair = make_dpo_pair(ex, strategy=None)

            if pair:
                dpo_pairs.append(pair)
                strategy_counter[pair['strategy_used']] += 1
            else:
                skipped += 1

    return dpo_pairs, strategy_counter, skipped


print('✅ DPO generator functions defined')
print('   make_dpo_pair() - converts 1 SFT example to 1 DPO pair')
print('   generate_dpo_dataset() - converts a full split')


✅ DPO generator functions defined
   make_dpo_pair() - converts 1 SFT example to 1 DPO pair
   generate_dpo_dataset() - converts a full split


---
## Step 5 — Generate DPO Pairs for All Splits

In [6]:
print('Generating DPO pairs...')
print('(Each SFT example → 1 DPO pair with rotated rejection strategy)')
print()

# Generate for each split
print('📋 Processing train split...')
dpo_train, counter_train, skip_train = generate_dpo_dataset(train_sft, n_per_example=1)
print(f'   ✅ {len(dpo_train)} pairs generated, {skip_train} skipped')

print('📋 Processing val split...')
dpo_val, counter_val, skip_val = generate_dpo_dataset(val_sft, n_per_example=1)
print(f'   ✅ {len(dpo_val)} pairs generated, {skip_val} skipped')

print('📋 Processing test split...')
dpo_test, counter_test, skip_test = generate_dpo_dataset(test_sft, n_per_example=1)
print(f'   ✅ {len(dpo_test)} pairs generated, {skip_test} skipped')

total = len(dpo_train) + len(dpo_val) + len(dpo_test)
print(f'\n📊 TOTAL DPO PAIRS: {total}')
print(f'   dpo_train: {len(dpo_train)}')
print(f'   dpo_val:   {len(dpo_val)}')
print(f'   dpo_test:  {len(dpo_test)}')


Generating DPO pairs...
(Each SFT example → 1 DPO pair with rotated rejection strategy)

📋 Processing train split...
   ✅ 1615 pairs generated, 1 skipped
📋 Processing val split...
   ✅ 202 pairs generated, 0 skipped
📋 Processing test split...
   ✅ 202 pairs generated, 1 skipped

📊 TOTAL DPO PAIRS: 2019
   dpo_train: 1615
   dpo_val:   202
   dpo_test:  202


---
## Step 6 — Check Strategy Distribution

A good DPO dataset has **even distribution** across rejection types — no single strategy should dominate.

In [7]:
print('📊 Strategy distribution in training set:')
print('=' * 60)
total_train = sum(counter_train.values())
for strat, count in sorted(counter_train.items(), key=lambda x: -x[1]):
    pct = count / total_train * 100
    bar = '█' * int(pct // 2)
    print(f'  {strat:<28} {count:>5} ({pct:5.1f}%)  {bar}')

print()
print('✅ Good distribution: each strategy ~7-12% of total')
print('⚠️  If any strategy > 25%, regenerate with strategy_rotation=True')


📊 Strategy distribution in training set:
  hallucinated_fields            151 (  9.3%)  ████
  invalid_json_string            151 (  9.3%)  ████
  partial_extraction             151 (  9.3%)  ████
  extra_text_wrapper             148 (  9.2%)  ████
  missing_fields                 148 (  9.2%)  ████
  wrong_values                   147 (  9.1%)  ████
  nested_wrong                   147 (  9.1%)  ████
  wrong_key_names                146 (  9.0%)  ████
  empty_values                   145 (  9.0%)  ████
  swapped_values                 145 (  9.0%)  ████
  type_errors                    136 (  8.4%)  ████

✅ Good distribution: each strategy ~7-12% of total
⚠️  If any strategy > 25%, regenerate with strategy_rotation=True


---
## Step 7 — Validate DPO Pairs

Every pair is checked:
- `prompt` is not empty
- `chosen` != `rejected` (the model has something to learn)
- `chosen` is valid JSON
- Both fields are strings

In [8]:
def validate_dpo_pairs(pairs, split_name=''):
    valid = []
    issues = []

    for i, pair in enumerate(pairs):
        errs = []

        # Check prompt
        if not pair.get('prompt', '').strip():
            errs.append('empty prompt')

        # Check chosen and rejected are strings
        if not isinstance(pair.get('chosen'), str):
            errs.append('chosen is not a string')
        if not isinstance(pair.get('rejected'), str):
            errs.append('rejected is not a string')

        # Check they are different
        if pair.get('chosen') == pair.get('rejected'):
            errs.append('chosen == rejected (no learning signal)')

        # Check chosen is valid JSON
        try:
            json.loads(pair['chosen'])
        except:
            errs.append('chosen is not valid JSON')

        # Check prompt length is reasonable
        if len(pair.get('prompt','')) > 1000:
            errs.append('prompt very long (> 1000 chars)')

        if errs:
            issues.append((i, errs, pair))
        else:
            valid.append(pair)

    print(f'{split_name}: {len(valid)} valid, {len(issues)} invalid')
    if issues:
        print('  Sample issues:')
        for idx, errs, p in issues[:3]:
            print(f'    Row {idx}: {errs}')
            print(f'    Prompt: {p.get("prompt","?")[:60]}')

    return valid


print('🔍 Validating DPO pairs...')
dpo_train_valid = validate_dpo_pairs(dpo_train, 'dpo_train')
dpo_val_valid   = validate_dpo_pairs(dpo_val,   'dpo_val')
dpo_test_valid  = validate_dpo_pairs(dpo_test,  'dpo_test')

print(f'\n✅ Total valid pairs: {len(dpo_train_valid) + len(dpo_val_valid) + len(dpo_test_valid)}')


🔍 Validating DPO pairs...
dpo_train: 1615 valid, 0 invalid
dpo_val: 202 valid, 0 invalid
dpo_test: 202 valid, 0 invalid

✅ Total valid pairs: 2019


---
## Step 8 — Preview Sample Pairs

Inspect a few examples from each rejection category.

In [9]:
# Show 1 example per strategy type
strategy_seen = {}
for pair in dpo_train_valid:
    s = pair.get('strategy_used')
    if s and s not in strategy_seen:
        strategy_seen[s] = pair

print('📋 One example per rejection strategy:')
print('=' * 70)
for strat, pair in strategy_seen.items():
    print(f'\n🔴 Strategy: [{strat}]')
    print(f'  PROMPT:   {pair["prompt"]}')
    print(f'  CHOSEN:   {pair["chosen"]}')
    print(f'  REJECTED: {pair["rejected"]}')
    print(f'  → Why rejected is worse: ', end='')
    explanations = {
        'missing_fields': 'dropped important fields from the extraction',
        'wrong_key_names': 'used incorrect field names (capitalised/aliased)',
        'wrong_values': 'corrupted a value (typo, wrong number)',
        'hallucinated_fields': 'added fields that were NOT in the input text',
        'partial_extraction': 'extracted only 1 field, ignored the rest',
        'type_errors': 'returned a number as a string (wrong data type)',
        'invalid_json_string': 'produced malformed / unparseable JSON',
        'extra_text_wrapper': 'wrapped JSON in explanation text (not pure JSON)',
        'swapped_values': 'mixed up values between wrong fields',
        'empty_values': 'left field values empty or null',
        'nested_wrong': 'incorrectly nested values inside sub-objects',
    }
    print(explanations.get(strat, 'general quality issue'))


📋 One example per rejection strategy:

🔴 Strategy: [extra_text_wrapper]
  PROMPT:   Aaron Mehta is a 37-year-old civil engineer living in Lima.
  CHOSEN:   {"name": "Aaron Mehta", "age": 37, "job": "civil engineer", "city": "Lima"}
  REJECTED: Extraction result: {"name": "Aaron Mehta", "age": 37, "job": "civil engineer", "city": "Lima"}. Hope this helps!
  → Why rejected is worse: wrapped JSON in explanation text (not pure JSON)

🔴 Strategy: [empty_values]
  PROMPT:   Customer Zara Ali placed order #52914 for 10 units of green mechanical keyboard at $1544.96/unit. Total: $15449.6.
  CHOSEN:   {"customer": "Zara Ali", "order_id": 52914, "product": "mechanical keyboard", "color": "green", "quantity": 10, "price_per_unit": 1544.96, "total": 15449.6}
  REJECTED: {"customer": "Zara Ali", "order_id": 52914, "product": "N/A", "color": "green", "quantity": 10, "price_per_unit": null, "total": 15449.6}
  → Why rejected is worse: left field values empty or null

🔴 Strategy: [wrong_values]
  PROM

---
## Step 9 — Save DPO Dataset

Two formats saved:
1. **With `strategy_used`** → useful for analysis and debugging
2. **Clean (without `strategy_used`)** → ready for TRL DPOTrainer

In [10]:
def save_dpo_jsonl(filename, pairs, include_strategy=True):
    path = os.path.join(OUT_DIR, filename)
    with open(path, 'w', encoding='utf-8') as f:
        for pair in pairs:
            if include_strategy:
                f.write(json.dumps(pair) + '\n')
            else:
                # Clean version: only prompt, chosen, rejected
                clean = {k: v for k, v in pair.items() if k != 'strategy_used'}
                f.write(json.dumps(clean) + '\n')
    size_kb = os.path.getsize(path) / 1024
    print(f'  ✅ {filename:<30} {len(pairs):>5} pairs   {size_kb:.1f} KB')


print('Saving DPO dataset files...')
print()

# Save with strategy info (for analysis)
print('📁 Files with strategy_used field (for analysis):')
save_dpo_jsonl('dpo_train.jsonl',         dpo_train_valid, include_strategy=True)
save_dpo_jsonl('dpo_val.jsonl',           dpo_val_valid,   include_strategy=True)
save_dpo_jsonl('dpo_test.jsonl',          dpo_test_valid,  include_strategy=True)

print()
# Save clean versions (for actual training)
print('📁 Clean files for TRL DPOTrainer (no strategy field):')
save_dpo_jsonl('dpo_train_clean.jsonl',   dpo_train_valid, include_strategy=False)
save_dpo_jsonl('dpo_val_clean.jsonl',     dpo_val_valid,   include_strategy=False)
save_dpo_jsonl('dpo_test_clean.jsonl',    dpo_test_valid,  include_strategy=False)

print(f'\n✅ All saved to: {os.path.abspath(OUT_DIR)}')


Saving DPO dataset files...

📁 Files with strategy_used field (for analysis):
  ✅ dpo_train.jsonl                 1615 pairs   640.2 KB
  ✅ dpo_val.jsonl                    202 pairs   78.0 KB
  ✅ dpo_test.jsonl                   202 pairs   82.3 KB

📁 Clean files for TRL DPOTrainer (no strategy field):
  ✅ dpo_train_clean.jsonl           1615 pairs   583.4 KB
  ✅ dpo_val_clean.jsonl              202 pairs   70.9 KB
  ✅ dpo_test_clean.jsonl             202 pairs   75.2 KB

✅ All saved to: c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\data


---
## Step 10 — Final Sanity Check

In [11]:
print('🔍 Reloading and verifying saved files...')

for fname in ['dpo_train_clean.jsonl', 'dpo_val_clean.jsonl', 'dpo_test_clean.jsonl']:
    path = os.path.join(OUT_DIR, fname)
    rows = []
    errors = 0
    with open(path, encoding='utf-8') as f:
        for line in f:
            r = json.loads(line.strip())
            rows.append(r)
            # Verify required keys
            if not all(k in r for k in ['prompt','chosen','rejected']):
                errors += 1
            # Verify chosen != rejected
            if r['chosen'] == r['rejected']:
                errors += 1

    status = '✅' if errors == 0 else f'❌ {errors} errors'
    print(f'  {fname:<35} {len(rows):>5} rows  {status}')

print()
print('📋 Sample clean DPO pair (what TRL DPOTrainer receives):')
sample_path = os.path.join(OUT_DIR, 'dpo_train_clean.jsonl')
with open(sample_path, encoding='utf-8') as f:
    sample = json.loads(f.readline())
print(json.dumps(sample, indent=2))


🔍 Reloading and verifying saved files...
  dpo_train_clean.jsonl                1615 rows  ✅
  dpo_val_clean.jsonl                   202 rows  ✅
  dpo_test_clean.jsonl                  202 rows  ✅

📋 Sample clean DPO pair (what TRL DPOTrainer receives):
{
  "prompt": "Aaron Mehta is a 37-year-old civil engineer living in Lima.",
  "chosen": "{\"name\": \"Aaron Mehta\", \"age\": 37, \"job\": \"civil engineer\", \"city\": \"Lima\"}",
  "rejected": "Extraction result: {\"name\": \"Aaron Mehta\", \"age\": 37, \"job\": \"civil engineer\", \"city\": \"Lima\"}. Hope this helps!"
}
